<a href="https://colab.research.google.com/github/maulikcmr05/NLP/blob/main/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
from transformers import pipeline

In [2]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [4]:
def get_wordnet_pos(tag):
  if tag.startswith("J"):
    return wordnet.ADJ
  elif tag.startswith("V"):
    return wordnet.VERB
  elif tag.startswith("N"):
    return wordnet.NOUN
  elif tag.startswith("R"):
    return wordnet.ADV
  else:
    return wordnet.NOUN

In [5]:
text = input("Enter a sentence: ")

print("\n========== TOKENIZATION ==========")
tokens = word_tokenize(text)
print(tokens)

Enter a sentence: Akash is so lazy

========== TOKENIZATION ==========
['Akash', 'is', 'so', 'lazy']


In [6]:
print("\n========== MORPHOLOGICAL ANALYSIS ==========")
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
pos_tags = pos_tag(tokens)
print("{:<15}{:<15}{:<15}".format("Word", "Stem", "Lemma"))

print("-" * 45)
for word, tag in pos_tags:
  stem = stemmer.stem(word)
  lemma = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
  print("{:<15}{:<15}{:<15}".format(word, stem, lemma))


========== MORPHOLOGICAL ANALYSIS ==========
Word           Stem           Lemma          
---------------------------------------------
Akash          akash          Akash          
is             is             be             
so             so             so             
lazy           lazi           lazy           


In [7]:
print("\n========== SYNTACTIC ANALYSIS ==========")
print("{:<15}{:<10}".format("Word", "POS Tag"))
print("-" * 30)
for word, tag in pos_tags:
  print("{:<15}{:<10}".format(word, tag))


========== SYNTACTIC ANALYSIS ==========
Word           POS Tag   
------------------------------
Akash          NNP       
is             VBZ       
so             RB        
lazy           JJ        


In [8]:
print("\n========== SEMANTIC ANALYSIS ==========")
for word in tokens:
  synsets = wordnet.synsets(word)
  if synsets:
      print(f"{word:<15} : {synsets[0].definition()}")
  else:
    print(f"{word:<15} : No meaning found")


========== SEMANTIC ANALYSIS ==========
Akash           : No meaning found
is              : have the quality of being; (copula, used with an adjective or a predicate noun)
so              : the syllable naming the fifth (dominant) note of any musical scale in solmization
lazy            : moving slowly and gently


In [9]:
print("\n========== PRAGMATIC ANALYSIS ==========")
# Load model only once
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)
sentence = text.strip().lower()
# Keywords
request_words = [
    "please", "could you", "can you",
    "would you", "will you",
    "kindly", "may i", "would you mind"
]

command_words = [
    "open", "close", "run", "execute",
    "print", "show", "display",
    "write", "train", "stop",
    "start", "read", "save",
    "delete", "install",
    "create", "calculate",
    "find", "search"
]


========== PRAGMATIC ANALYSIS ==========


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [10]:
# Rule-based classification
if sentence.endswith("?"):
  if any(word in sentence for word in request_words):
    intent = "Request"
    confidence = 1.0000
  else:
      intent = "Question"
      confidence = 1.0000
elif any(sentence.startswith(word) for word in command_words):
  intent = "Command"
  confidence = 1.0000
elif any(word in sentence for word in request_words):
  intent = "Request"
  confidence = 1.0000
else:
  # Zero-shot fallback
  labels = [
      "This sentence is a statement.",
      "This sentence is a question.",
      "This sentence is a request.",
      "This sentence is a command."
      ]
  result = classifier(
      text,
      candidate_labels=labels,
      hypothesis_template="{}"
      )
  intent = result["labels"][0].replace("This sentence is a ","").replace(".", "").title()
  confidence = result["scores"][0]

print("Predicted Intent :", intent)
print("Confidence Score :", round(confidence, 4))

Predicted Intent : Statement
Confidence Score : 0.5321
